In [ ]:
%%capture
!pip install transformers>=4.41.2 accelerate>=0.31.0

lets see how the transformer language models work, our focus will be on text generation models so we get a deeper sense of whats happening in generative LLMs

load the model

device_map="cuda" tells where to load the model, cuda loads the entire model on the gpu,

torch_dtype="auto", it decides the numerical precision the models use, the more the no the slower the model, "auto" can reduce the memory usage if the model supports lower precision

trust_remote_code=False, this prevents download of custom python code present in hugging face models used for custom models

return_full_text=False, ensure the prompt doesnt returns in the output

do_sample=False, allows the model to generate same outputs in all the runs, sampling basically allows the model to be more creative and generate responses in more different ways

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


AN OVERVIEW OF TRANSFORMER MODELS

we will take the in depth overview of the transformer model introduced in 2017 and see how the recent works have improved it

THE INPUTS AND OUTPUTS OF THE TRAINED TRANSFORMER LLM

the most common way to look at LLMs is imagining it as a software that takes text input and generated text in response, and when we train a large text in text out model using a high quality dataset we get a model that can generate impressive and useful outputs, fig 3-1 shows such modle used to mail an author

the model dont generate the text all in once, but instead one token a time, fig 3-2 shows the four steps of token gen, where each token generation step is one forward pass through the model, just like ML where inputs goes into the network and after computations it needs to produce an output at the other end of computation graph <br>
now after each token generation we tweak the input a little by adding the gen output token at the end of input prompt, seen in fig 3-3

this gives a clear picture of the model as it is simply predicting the next token based on the input prompt, the software around these neural networks runs this in loops so we can sequentially expand the generated text until completion

the models which use their earlier predictions to make next predictions are called autoregressive models in ML, this is why text generative models are called auto regressive models, this is used to diff between text generation and text representation models like BERT which is not autoregressive

now we wil gen text with LLM, token by token like we see here..

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


the model wrote the email and stopped in between because it reached the max new tokens limit which was set to 50, if we increase it, the model will complete the response

THE COMPONENTS OF THE FORWARD PASS

in addition to the loop we have two more imp components, the tokenizer and the language modeling head (LM head), fig 3-4 shows where are these components present in the architecture, in the previous chapters we have seen how the tokenizer break down the input and transform it to seq of token ids that becomes the input to the model

after tokenizer we have a neural network which is the stack of transformer blocks that do all the processing, this stack is followed by LM head which converts the output of the stack to probab scores to decide what next token is

the tokenizer contains a table of tokens which is the tokenizer's vocab and the model contains vector representations for each of these tokens called token embeddings stored in its vocab, fig 3-5 shows a model with a vocab of 50000 tokens and the embeddings associated with it

the flow of computation follow the direction of the arrow in fig, for each gen token the process flows once through each of these blocks in order then to LM head where the outputs are converted to probab dist to decide the next token to be produced from the tokenizers vocab as seen in fig 3-6

the LM head we have is a neural network itself, it is one of the "heads" which one can attach to the stack of transformers to build diff type of systems, other heads we have are sequence classification heads and token classification heads

we can actually see the model layers by printing the model variable

In [ ]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

this structure tells us following things about the model arch :

1. there are various nested layers in the model, the model is named "model" and its followed by an LM head

2. inside the Phi3Model we can see the embeddings matrix embed_tokens, it has 32064 tokens and each has a vector size of 3072

3. the next major component is stack of transformer decode layer where we have 32 blocks of type Phi3DecodeLayer

4. now for each of these blocks we have an attention layer and a feed forward neural network also known as mlp or multi layer perceptron

5. at last we can see the lm head taking the vector of 3072 and producing an output vector of size equivalent to the no of tokens the model knows, that output is basically the probab score for each token, this will help us to select the next token

CHOOSING A SINGLE TOKEN FROM THE PROBABILITY DISTRIBUTION (SAMPLING/DECODING)


at the end of the processing we gets an probab dist to select the tokens this is called decoding stratgy, fig 3-7 shows the process of picking dear for eg

the best way would be simply picking the token with highest probability score, but in practice it doesnt yield good results always, what we need is to add randomness sometimes in the selection like selecting the third or second highest probab token, this called sample from the probab dist based on the probab score

for the eg in fig 3-7, if the token "dear" have 40 % probab, it would mean 40% chance of it getting selected, but in greedy method it would already have been selected for having the highest probab, sampling gives chance of being picked according to the score

the greedy decoding can be implemented by settin the temp param to zero

the below code shows us how to do it, the input token is passed to the model and then lm_head

In [ ]:
prompt = "The capital of France is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Tokenize the input prompt
input_ids = input_ids.to("cuda")

# Get the output of the model before the lm_head
model_output = model.model(input_ids)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

the lm_head output is of shape [1,6,32064],

1 is for batch which says i passed one sentence as input if passed 8 it would be 8, 6 is for sequence this tells us the no of tokens in input and 32064 is vocab size or the no of tokens

we can get the last generated token probab by using lm_head_output[0,-1], it uses the index 0 across the batch dimension which means take the first and only batch, the index -1 is used to get to the last token in sequence coz we want the token next to it be predicted

lm_head_output <br>
│<br>
├── batch 0<br>
│<br>
│   token 0 → 32064 scores<br>
│   token 1 → 32064 scores<br>
│   token 2 → 32064 scores<br>
│   token 3 → 32064 scores<br>
│   token 4 → 32064 scores<br>
│   token 5 → 32064 scores  ← we want this one

all this is a list of probab scores for 32.064 tokens and we can get the top scoring token id and then decode it to get the text of generated output token, in this case we have paris as the output

In [ ]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

PARALLEL TOKEN PROCESSING AND CONTEXT SIZE

one of the best feature of LLM is their compatibility with parallel computing which is better than previous neural network arch in language processing. we get to know more about this when we look at how text is processed in LLMs, we know the input text is converted to tokens and each of these tokens follows their own path of computation, fig 3-8 shows the individual processing tracks or streams

the current transformer model has a limit for how many tokens they can process at once. that limit is called model's context length

each of these token streams starts with an input vector or embedding vector, and some positional info is also included , and at the end we get another vector as the result of models processing shown in fig 3-9

for text generation only the output of the last token stream is used to predict the next token, that output vector is the only input in the lm head as it calculates the probab of next token

one may think why we are calculating other streams output if we are not using them, the ans is that the calculations of previous streams is required to generate the final stream's output, yes we may not be using their final output but we use their earlier outputs from each transformer block into the transformers block attention mechanism

the output of the lm head was [1,6,32064] this was because the input was of shape [1,6,3072] a batch of one input string containing six tokens of vector size 3072 each which corresponds to the output vectors after the stack of transformer blocks

these matrices can be accessed by printing :

In [ ]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [ ]:
lm_head_output.shape

torch.Size([1, 5, 32064])

SPEEDING UP GENERATION BY CACHING KEYS AND VALUES

when we generate the second token we append the previous generated 1st token to the input and run a forward pass again, but if we cache the results of the previous calculation we do not need to calculate it in future only the final stream output will be calculated, this is an optimization technique called keys and values cache (kv)cache and it speeds up the generation process, they are one of the central components in attention mechanism

fig 3-10 shows the process where we cached the results of previous streams and now only one stream is active

the hugging face transformers enables caching by default and we can disable it by setting use_cache to false and we can see the difference in speed by asking for long generation and timing the generation with and without caching

In [ ]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

now we can see how long it takes to generate 100 tokens with caching, we can use %%timeit command to see how long the execution takes, this command will run the execution several times and outputs the average

In [ ]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=True
)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


5.24 s ± 337 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=False
)

33.5 s ± 207 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


we can see a dramatic difference in time required to generate 100 tokens,

the 5.24 s we got with caching is also high in users point of view, this wy LLM api's stream the output token as the model generates output instead of waiting for the full generation to be completed

INSIDE THE TRANSFORMER BLOCK

the majority of the processing happens inside the transformer blocks, the original paper had 6 but it expands to 100's in large llms, each block process the input and passes its output as an input to other block, fig 3-11 shows this

now look at the fig 3-12 we have two main components for each block

1. the attention layer : which incorporates the information related to other input tokens and positions
2. the feedforward layer : which includes the majority of the models processing capacity

the feed forward neural network

a simple ex would be , if we pass the input "the shawshank" to the llm we would expect it to return "redemption" in return as it will be the most probable next word (1994 film)

the feed forward neural network is the reason we got this output as when we train a model to model the massive text archive which already have mentions of the shawshank redemption, it learned and stored the info and behavior that make it succeed this task see fig 3-13

for an llm to perform well one may consider memorization as an essential skill for the model, as the more the model remembers the bettter it would be, but it is not the only key for impressive text generation, the model should be able to interpolate between data points and more complex patterns so that it generalizes well on unseen data or the inputs which it had'nt seen in past and were not present in its training set

note:

when we use todays modern llms we dont get the outputs mentioned earlier, a simpler "the shawshank" may return

"The Shawshank Redemption" is a 1994 film directed by
Frank Darabont and is based on the novella "Rita
Hayworth and Shawshank Redemption" written by Stephen
King. ...etc.

this is because the raw models like gpt3 is difficult to be used by common ppl, so the new models like gpt 4 uses instruction tuning and human preference and feedback fine tuning to make the responses better for everyone according to their expectations